# Gravitational N-body problem — quick start

**Author:** [Jesús García Izquierdo](https://github.com/jesus-garcia-izquierdo) · Repository: [N-body-problem](https://github.com/jesus-garcia-izquierdo/N-body-problem)

This notebook runs in Google Colab without any local installation. It:

1. integrates the **figure-eight choreography** of three equal masses with the symplectic velocity Verlet method,
2. checks the **conservation laws** (energy, linear and angular momentum) and the **virial theorem**,
3. renders a **3D video** of the orbit,
4. shows **chaos**: two six-body systems that differ by 0.1 % in one coordinate.

The physics is explained in [docs/THEORY.md](https://github.com/jesus-garcia-izquierdo/N-body-problem/blob/main/docs/THEORY.md) and in the PDF reports.

In [ ]:
# Get the code and install the package (numpy and matplotlib are already in Colab)
!git clone -q https://github.com/jesus-garcia-izquierdo/N-body-problem.git
%cd N-body-problem
!pip install -q -e .

## 1. Figure-eight choreography

N-body units: $G = 1$, $m_i = 1$, period $T \approx 6.326$. The step $h = T/6000$ is far below the shortest dynamical time.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from nbody import scenarios, simulate, diagnostics, plotting, chaos

T = scenarios.FIGURE_EIGHT_PERIOD
system = scenarios.figure_eight()                     # NBodySystem: masses (3,), positions (3, 3), velocities (3, 3)
traj = simulate(system, t_final=5 * T, h=T / 6000, integrator="verlet", save_every=6)
print(diagnostics.conservation_report(traj, "Figure-eight"))

In [ ]:
# Orbits and conservation laws (energy error bounded: Verlet is symplectic)
plotting.plot_orbits(traj, projection="xy", unit_label="length units", title="Figure-eight (5 periods)")
plotting.plot_conservation(traj, time_unit=T, time_label="periods")
plt.show()

## 2. 3D video

`render_video` draws every frame with a rotating camera and fading trails and encodes an MP4 with ffmpeg (preinstalled in Colab).

In [ ]:
from base64 import b64encode
from IPython.display import HTML
from nbody.video import render_video

clip = simulate(system, t_final=2 * T, h=T / 3000, save_every=2)
render_video(clip, "figure_eight.mp4", title="Figure-eight choreography", time_unit=T,
             time_label="periods", duration=8, trail=T / 3, sizes=[80] * 3, elev=30, zoom=1.35)

data = b64encode(open("figure_eight.mp4", "rb").read()).decode()
HTML(f'<video width="720" controls autoplay loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## 3. Chaos: a 0.1 % change in one initial condition

Six bodies of $10^{10}$ kg. The separation between the two runs grows exponentially (a straight line on a log scale) until it saturates at the size of the system.

In [ ]:
from nbody import constants as c

base = scenarios.six_body_report()
perturbed = base.perturbed(body=0, quantity="position", component=0, relative=1e-3)
ref = simulate(base, 6 * c.YEAR, 1000.0, save_every=50)
pert = simulate(perturbed, 6 * c.YEAR, 1000.0, save_every=50)
delta = chaos.position_divergence(ref, pert)
plotting.plot_divergence(ref.t, {"x₁ + 0.1 %": delta}, time_unit=c.YEAR, time_label="years",
                         reference_level=(1e5, "size of the system"))
plt.show()

## Next steps

* Run the eight examples in `examples/` (`!python examples/07_solar_system.py --no-video`).
* Change the initial conditions with `NBodySystem(masses, positions, velocities, G)`.
* Compare integrators: `integrator="euler" | "symplectic_euler" | "verlet" | "yoshida4" | "rk4"`.